In [1]:
import pandas as pd
import plotly.express as px
import os

# --- Data Loading ---
DATA_PATH = '../../data/myanmar_conflict_clean.csv'
if not os.path.exists(DATA_PATH):
    print(f'Error: Data file not found at {DATA_PATH}')
else:
    df = pd.read_csv(DATA_PATH)
    df['event_date'] = pd.to_datetime(df['event_date'])
    df = df[df['event_date'] >= '2021-02-01']

    # --- Helper Function ---
    def categorize_actor(actor_name):
        if pd.isna(actor_name): return 'Unidentified'
        name = str(actor_name).lower()
        if any(x in name for x in ['military forces of myanmar', 'police forces of myanmar', 'state administration council', 'border guard force', 'people\'s militia force']):
            return 'State Forces'
        elif any(x in name for x in ['pyu saw htee', 'thway thauk', 'blood comrades', 'swan arr shin', 'pro-military']):
            return 'Pro-Junta Militia'
        resistance_keywords = ['pdf', 'people\'s defense force', 'local defense force', 'kndf', 'karenni nationalities defense force', 'chinland defense force', 'cdf', 'guerrilla', 'resistance', 'attack force', 'generation z', 'gen z', 'drone strike', 'militia', 'security force', 'freedom force', 'tiger force', 'ranger group', 'truth army']
        if any(x in name for x in resistance_keywords):
            return 'Resistance'
        eao_keywords = ['knu', 'knla', 'kndo', 'kia', 'kio', 'tnla', 'pslf', 'mndaa', 'mntjp', 'aa ', 'ula', 'rcss', 'ssa', 'knpp', 'ka ', 'cnf', 'cna', 'pno', 'brotherhood alliance', 'northern alliance', 'three brotherhood', 'absdf', 'kachin', 'karen', 'shan state', 'arakan', 'mon state', 'nmsp', 'nmla', 'chin brotherhood', 'rohingya', 'arsa', 'kaw thoo lei', 'pa-oh', 'ta\'ang', 'palaung', 'kokang', 'wa state', 'uwsa', 'naga', 'nscn', 'ktla']
        if any(x in name for x in eao_keywords):
            return 'EAOs'
        if 'protesters' in name: return 'Protesters'
        elif 'rioters' in name: return 'Rioters'
        elif 'civilians' in name: return 'Civilians'
        if 'unidentified' in name: return 'Unidentified'
        else: return 'Other Groups'

    # --- Categorize ---
    df['Actor_Category'] = df['actor1'].apply(categorize_actor)
    fatal_events = df[df['fatalities'] > 0].copy()

    # --- Generate Plot ---
    fig = px.box(
        fatal_events, 
        x='Actor_Category', 
        y='fatalities', 
        color='Actor_Category',
        title='Distribution of Fatalities per Event by Actor Category',
        labels={'fatalities': 'Number of Fatalities', 'Actor_Category': 'Actor Category'},
        color_discrete_map={
            'State Forces': '#1e293b', 'Pro-Junta Militia': '#991b1b', 'Resistance': '#10b981', 
            'EAOs': '#0369a1', 'Civilians': '#f59e0b', 'Protesters': '#8b5cf6', 
            'Rioters': '#f97316', 'Unidentified': '#475569', 'Other Groups': '#64748b'
        }
    )
    
    fig.update_layout(
        paper_bgcolor='white', 
        plot_bgcolor='white',
        showlegend=False,
        yaxis=dict(type='log')
    )
    
    fig.show()
